In [9]:
import tensorflow as tf
print(tf.__file__)
print(tf.__version__)

d:\Python\Python38\lib\site-packages\tensorflow\__init__.py
2.10.0


In [10]:
import tensorflow as tf
from tensorflow.examples.tutorials.mnist import input_data

tf.compat.v1.disable_eager_execution()
mnist = input_data.read_data_sets('MNIST_data', one_hot=True)
tf.compat.v1.set_random_seed(1234)

def weight_variable(shape):
    return tf.Variable(tf.random.truncated_normal(shape, stddev=0.1))

def bias_variable(shape):
    return tf.Variable(tf.constant(0.1, shape=shape))

def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1,1,1,1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool(x,
                          ksize=[1,2,2,1],
                          strides=[1,2,2,1],
                          padding='SAME')

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 10000

xs = tf.compat.v1.placeholder(tf.float32, [None,784])
ys = tf.compat.v1.placeholder(tf.float32, [None,10])
keep_prob = tf.compat.v1.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1,28,28,1])

W_conv1 = weight_variable([7,7,1,32]); b_conv1=bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1)+b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

W_conv2 = weight_variable([5,5,32,64]); b_conv2=bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1,W_conv2)+b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

W_fc1 = weight_variable([7*7*64, 1024]); b_fc1=bias_variable([1024])
h_pool2_flat = tf.reshape(h_pool2, [-1,7*7*64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat,W_fc1)+b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, rate=1-keep_prob)

W_fc2 = weight_variable([1024,10]); b_fc2=bias_variable([10])
logits = tf.matmul(h_fc1_drop,W_fc2)+b_fc2
prediction = tf.nn.softmax(logits)

loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(labels=ys, logits=logits))
train_step = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(loss)

correct_prediction = tf.equal(tf.argmax(prediction,1), tf.argmax(ys,1))
acc_op = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))

with tf.compat.v1.Session() as sess:
    sess.run(tf.compat.v1.global_variables_initializer())

    for i in range(max_epoch):
        bx, by = mnist.train.next_batch(100)
        bx = bx / 255.0
        _, loss_val = sess.run([train_step, loss], feed_dict={xs:bx, ys:by, keep_prob:keep_prob_rate})

        if i % 500 == 0:
            testx = mnist.test.images[:1000] / 255.0
            testy = mnist.test.labels[:1000]
            a = sess.run(acc_op, feed_dict={xs:testx, ys:testy, keep_prob:1.0})
            print(f"Step {i}, loss = {loss_val:.4f}, acc = {a:.4f}")

    testx = mnist.test.images / 255.0
    testy = mnist.test.labels
    final_acc = sess.run(acc_op, feed_dict={xs:testx, ys:testy, keep_prob:1.0})
    print(f"Final test accuracy = {final_acc:.4f}")

Extracting MNIST_data\train-images-idx3-ubyte.gz
Extracting MNIST_data\train-labels-idx1-ubyte.gz
Extracting MNIST_data\t10k-images-idx3-ubyte.gz
Extracting MNIST_data\t10k-labels-idx1-ubyte.gz
Step 0, loss = 5.5510, acc = 0.0850
Step 500, loss = 2.2908, acc = 0.1750
Step 1000, loss = 1.9093, acc = 0.5060
Step 1500, loss = 0.9151, acc = 0.7620
Step 2000, loss = 0.6897, acc = 0.8290
Step 2500, loss = 0.5024, acc = 0.8660
Step 3000, loss = 0.3423, acc = 0.8790
Step 3500, loss = 0.4096, acc = 0.8860
Step 4000, loss = 0.3440, acc = 0.8970
Step 4500, loss = 0.1475, acc = 0.9170
Step 5000, loss = 0.2924, acc = 0.9210
Step 5500, loss = 0.2883, acc = 0.9220
Step 6000, loss = 0.2638, acc = 0.9340
Step 6500, loss = 0.1801, acc = 0.9440
Step 7000, loss = 0.2517, acc = 0.9480
Step 7500, loss = 0.1422, acc = 0.9510
Step 8000, loss = 0.1368, acc = 0.9560
Step 8500, loss = 0.0320, acc = 0.9590
Step 9000, loss = 0.0998, acc = 0.9650
Step 9500, loss = 0.0979, acc = 0.9670
Final test accuracy = 0.9701
